# Narrative Shift Detection using Temporal Contrastive Learning (TCL)

## Stage-Wise Execution Pipeline

**Unified Model**: Single model trained on all 5 topics (Health, War, Technology, Climate, Economics)

**Key Innovation:**
- Entity-invariant embeddings (remove entity signal)
- Topic embeddings (64D) added AFTER grouping → 832D input
- Self-supervised temporal contrastive learning
- Multi-loss: temporal + topic separation + hard negatives

**Pipeline Flow:**
1. SBERT embedding (768D)
2. Entity extraction + cleaning
3. Day-level aggregation (training) / Article-level (inference)
4. Ruptures grouping (training only)
5. Topic embedding addition (768D → 832D)
6. Window creation
7. Temporal Transformer (832D → 256D)
8. Multi-objective loss training

**Hardware:** Kaggle GPU P100 (16GB VRAM)

---

---
# STAGE 0: Environment Setup
---

In [ ]:
# Install required packages
!pip install -q sentence-transformers transformers spacy scikit-learn ruptures
!python -m spacy download en_core_web_sm

print("✅ Packages installed")

In [ ]:
# Core imports
import os
import json
import pickle
import warnings
from pathlib import Path
from datetime import datetime
from collections import Counter
import copy

# Data processing
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ML/DL
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler

# NLP
import spacy
from sentence_transformers import SentenceTransformer

# Segmentation
import ruptures as rpt

# Viz
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print("✅ Imports loaded")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
# STAGE 1: Configuration
---

Define all hyperparameters and paths.

In [ ]:
class Config:
    """Central configuration for the entire pipeline."""
    
    # Data paths
    DATA_DIR = Path('/kaggle/input/datasets/prateek1005/distributed-data')
    OUTPUT_DIR = Path('/kaggle/working/outputs')
    CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
    
    # Topics
    TOPICS = ['Health', 'War', 'Technology', 'Climate', 'Economics']
    
    # Entity-invariant embedding
    ENTITY_LAMBDA = 0.3  # λ for subtraction method
    USE_PROJECTION = False  # Use projection-based removal?
    
    # Embedding
    SBERT_MODEL = 'all-mpnet-base-v2'  # 768-dim
    EMBEDDING_DIM = 768
    TOPIC_EMB_DIM = 64  # Topic embedding dimension
    CONCAT_DIM = 832  # 768 + 64
    
    # NER batch size
    NER_BATCH_SIZE = 256
    
    # Day-level aggregation
    AGGREGATION_METHOD = 'weighted_mean'
    
    # Grouping (training only)
    USE_RUPTURES = True
    RUPTURE_MODEL = 'rbf'
    RUPTURE_PEN = 10
    MIN_GROUP_SIZE = 5
    
    # Windowing
    WINDOW_SIZE = 3
    WINDOW_STRIDE = 1
    
    # Model architecture
    HIDDEN_DIM = 512
    NUM_HEADS = 8
    NUM_LAYERS = 4
    DROPOUT = 0.1
    OUTPUT_DIM = 256
    
    # Loss weights
    LAMBDA_TEMPORAL = 1.0
    LAMBDA_TOPIC_SEP = 0.3
    LAMBDA_HARD_NEG = 0.5
    TEMPERATURE = 0.07
    
    # Training
    BATCH_SIZE = 32
    NUM_EPOCHS = 50
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    GRAD_CLIP = 1.0
    
    # Mixed precision
    USE_AMP = True
    
    # Scheduler
    USE_COSINE_SCHEDULE = True
    WARMUP_EPOCHS = 5
    
    # Checkpointing
    SAVE_EVERY = 5
    
    # Inference
    SHIFT_THRESHOLD = 0.5
    
    # Device
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    def __init__(self):
        # Create directories
        self.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        self.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Initialize config
config = Config()

print("="*80)
print("CONFIGURATION LOADED")
print("="*80)
print(f"Device: {config.DEVICE}")
print(f"Topics: {config.TOPICS}")
print(f"Embedding: {config.SBERT_MODEL} ({config.EMBEDDING_DIM}D)")
print(f"Topic embedding: {config.TOPIC_EMB_DIM}D")
print(f"Concatenated input: {config.CONCAT_DIM}D")
print(f"Model output: {config.OUTPUT_DIM}D")
print(f"Window size: {config.WINDOW_SIZE}")
print(f"Batch size: {config.BATCH_SIZE}")
print(f"Epochs: {config.NUM_EPOCHS}")
print("="*80)

---
# STAGE 2: Data Preprocessing Functions
---

Define all preprocessing functions, then apply to each topic.

## 2.1: Helper Functions

In [ ]:
def parse_embedding_string(emb_str):
    """Parse embedding string to numpy array."""
    if isinstance(emb_str, np.ndarray):
        return emb_str
    if isinstance(emb_str, str):
        emb_str = emb_str.strip('[]').replace('\n', ' ')
        return np.fromstring(emb_str, sep=' ')
    return np.array(emb_str)

print("✅ Helper functions defined")

## 2.2: Load CSV Data

In [ ]:
def load_topic_csv(topic, data_dir):
    """
    Load CSV for a specific topic.
    
    Expected columns: date, sentence_id, main_sentence, w5_embedding (or embedding), {topic}
    Note: Topic weight column is just the topic name (e.g., 'Health', 'War'), not '{topic}_weight'
    """
    csv_path = data_dir / f"{topic}.csv"
    
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")
    
    print(f"Loading {topic} data from {csv_path}...")
    df = pd.read_csv(csv_path)
    
    print(f"  Loaded {len(df):,} rows")
    print(f"  Columns: {list(df.columns)}")
    
    # Parse date
    df['date'] = pd.to_datetime(df['date'])
    
    # Handle both 'w5_embedding' and 'embedding' column names
    if 'w5_embedding' in df.columns:
        print(f"  Renaming 'w5_embedding' to 'embedding'...")
        df = df.rename(columns={'w5_embedding': 'embedding'})
    
    # Parse embeddings
    if 'embedding' in df.columns:
        print(f"  Parsing embeddings...")
        df['embedding'] = df['embedding'].apply(parse_embedding_string)
    else:
        raise ValueError(f"No embedding column found! Available columns: {list(df.columns)}")
    
    # Sort by date
    df = df.sort_values('date').reset_index(drop=True)
    
    print(f"  Date range: {df['date'].min()} to {df['date'].max()}")
    print(f"  ✅ {topic} data loaded")
    
    return df

print("✅ CSV loading function defined")

## 2.3: Named Entity Recognition

In [ ]:
# Load spaCy model
print("Loading spaCy NER model...")
nlp = spacy.load('en_core_web_sm', disable=['parser', 'tagger', 'lemmatizer'])
print("✅ spaCy model loaded")

In [ ]:
def extract_entities_batch(df, batch_size=256):
    """
    Extract named entities from sentences using spaCy.
    
    Args:
        df: DataFrame with 'main_sentence' column
        batch_size: Batch size for spaCy processing
    
    Returns:
        df: DataFrame with added 'entities' column
    """
    print(f"Extracting entities (batch_size={batch_size})...")
    
    sentences = df['main_sentence'].tolist()
    all_entities = []
    
    for i in tqdm(range(0, len(sentences), batch_size), desc="NER"):
        batch_sentences = sentences[i:i+batch_size]
        docs = list(nlp.pipe(batch_sentences, batch_size=batch_size))
        
        for doc in docs:
            entities = [ent.text for ent in doc.ents]
            all_entities.append(entities)
    
    df['entities'] = all_entities
    
    # Stats
    num_with_entities = sum(1 for e in all_entities if len(e) > 0)
    total_entities = sum(len(e) for e in all_entities)
    
    print(f"✅ Entity extraction complete")
    print(f"   Sentences with entities: {num_with_entities}/{len(df)} ({num_with_entities/len(df)*100:.1f}%)")
    print(f"   Total entities: {total_entities:,}")
    print(f"   Avg entities/sentence: {total_entities/len(df):.2f}")
    
    return df

print("✅ NER function defined")

## 2.4: Entity Embeddings

In [ ]:
def compute_entity_embeddings(df, sbert_model):
    """
    Compute entity embeddings by encoding entity text.
    
    Args:
        df: DataFrame with 'entities' column
        sbert_model: SBERT model
    
    Returns:
        df: DataFrame with 'entity_embedding' column
    """
    print("Computing entity embeddings...")
    
    entity_embeddings = []
    embedding_dim = config.EMBEDDING_DIM
    
    for entities in tqdm(df['entities'], desc="Entity embeddings"):
        if len(entities) == 0:
            # No entities → zero vector
            entity_embedding = np.zeros(embedding_dim)
        else:
            # Concatenate all entity texts
            entity_text = " ".join(entities)
            # Encode
            entity_embedding = sbert_model.encode(entity_text, convert_to_numpy=True)
        
        entity_embeddings.append(entity_embedding)
    
    df['entity_embedding'] = entity_embeddings
    
    print(f"✅ Entity embeddings computed")
    
    return df

print("✅ Entity embedding function defined")

## 2.5: Entity-Invariant Cleaning

In [ ]:
def compute_entity_invariant_embeddings(df, method='subtraction', lambda_=0.3):
    """
    Compute entity-invariant embeddings.
    
    Args:
        df: DataFrame with 'embedding' and 'entity_embedding' columns
        method: 'subtraction' or 'projection'
        lambda_: Weight for subtraction method
    
    Returns:
        df: DataFrame with 'clean_embedding' column
    """
    print(f"Computing entity-invariant embeddings (method={method}, λ={lambda_})...")
    
    clean_embeddings = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Entity cleaning"):
        sem_emb = row['embedding']  # (768,)
        ent_emb = row['entity_embedding']  # (768,)
        
        if method == 'subtraction':
            # E_clean = E_sem - λ * E_ent
            clean_emb = sem_emb - lambda_ * ent_emb
        
        elif method == 'projection':
            # E_clean = E_sem - Proj(E_sem onto E_ent)
            ent_norm = np.linalg.norm(ent_emb)
            if ent_norm > 1e-6:
                projection = (np.dot(sem_emb, ent_emb) / (ent_norm ** 2)) * ent_emb
                clean_emb = sem_emb - projection
            else:
                clean_emb = sem_emb
        
        else:
            raise ValueError(f"Unknown method: {method}")
        
        # L2 normalize
        norm = np.linalg.norm(clean_emb)
        if norm > 1e-6:
            clean_emb = clean_emb / norm
        
        clean_embeddings.append(clean_emb)
    
    df['clean_embedding'] = clean_embeddings
    
    print(f"✅ Entity-invariant embeddings computed")
    
    return df

print("✅ Entity cleaning function defined")

## 2.6: Day-Level Aggregation

In [ ]:
def aggregate_to_day_level(df, topic):
    """
    Aggregate sentence embeddings to day-level using topic weights.
    
    Args:
        df: DataFrame with 'date', 'clean_embedding', '{topic}' columns
        topic: Topic name
    
    Returns:
        day_df: DataFrame with one row per day
    """
    print(f"Aggregating to day-level for {topic}...")
    
    # Weight column is just the topic name (e.g., 'Health', 'War', not 'Health_weight')
    weight_col = topic
    
    # Fallback to uniform weights if column missing
    if weight_col not in df.columns:
        print(f"  ⚠️  Column '{weight_col}' not found, using uniform weights")
        df[weight_col] = 1.0
    
    day_records = []
    
    for date, group in tqdm(df.groupby('date'), desc="Day aggregation"):
        embeddings = np.stack(group['clean_embedding'].values)
        weights = group[weight_col].values
        
        # Weighted average
        weighted_sum = (embeddings.T * weights).T.sum(axis=0)
        weight_sum = weights.sum()
        
        if weight_sum > 0:
            day_embedding = weighted_sum / weight_sum
        else:
            day_embedding = embeddings.mean(axis=0)
        
        # L2 normalize
        norm = np.linalg.norm(day_embedding)
        if norm > 1e-6:
            day_embedding = day_embedding / norm
        
        day_records.append({
            'date': date,
            'embedding': day_embedding,
            'num_sentences': len(group),
            'avg_weight': weights.mean()
        })
    
    day_df = pd.DataFrame(day_records)
    day_df = day_df.sort_values('date').reset_index(drop=True)
    
    print(f"✅ Aggregated to {len(day_df)} days")
    print(f"   Date range: {day_df['date'].min()} to {day_df['date'].max()}")
    print(f"   Avg sentences/day: {day_df['num_sentences'].mean():.1f}")
    
    return day_df

print("✅ Day aggregation function defined")

---
# STAGE 3: Grouping + Topic Embeddings
---

**Training pipeline**:
1. Detect ruptures (temporal grouping)
2. Group-level pooling
3. Add topic embeddings (768D → 832D)
4. Create windows

## 3.1: Ruptures Grouping

In [ ]:
def detect_ruptures(day_df, model='rbf', pen=10, min_size=5):
    """
    Detect change points in temporal embedding sequence.
    
    Args:
        day_df: DataFrame with 'embedding' column
        model: Rupture detection model
        pen: Penalty value (higher = fewer change points)
        min_size: Minimum size between change points
    
    Returns:
        day_df: DataFrame with added 'group' column
    """
    print(f"Detecting ruptures (model={model}, pen={pen}, min_size={min_size})...")
    
    # Validate input
    if len(day_df) < min_size:
        print(f"  ⚠️  Too few days ({len(day_df)} < {min_size}), creating single group")
        day_df['group'] = 0
        return day_df
    
    # Stack embeddings
    embeddings = np.stack(day_df['embedding'].values)
    
    # Detect change points
    algo = rpt.Pelt(model=model, min_size=min_size).fit(embeddings)
    change_points = algo.predict(pen=pen)
    
    if not change_points or len(change_points) == 0:
        print(f"  ⚠️  No change points detected, creating single group")
        day_df['group'] = 0
        return day_df
    
    print(f"   Detected {len(change_points)-1} change points")
    print(f"   Change points at indices: {change_points[:-1]}")
    
    # Assign groups
    groups = np.zeros(len(day_df), dtype=int)
    group_id = 0
    prev_cp = 0
    
    for cp in change_points:
        groups[prev_cp:cp] = group_id
        group_id += 1
        prev_cp = cp
    
    day_df['group'] = groups
    
    # Stats
    group_sizes = day_df.groupby('group').size()
    print(f"✅ Created {group_id} groups")
    print(f"   Group sizes: min={group_sizes.min()}, max={group_sizes.max()}, mean={group_sizes.mean():.1f}")
    
    return day_df

print("✅ Ruptures function defined")

## 3.2: Topic Embedding Functions

In [ ]:
def create_topic_mapping(topics):
    """
    Create topic to ID mapping.
    
    Args:
        topics: List of topic names
    
    Returns:
        topic_to_id: Dictionary mapping topic name to ID
        id_to_topic: Dictionary mapping ID to topic name
    """
    unique_topics = sorted(set(topics))
    topic_to_id = {topic: idx for idx, topic in enumerate(unique_topics)}
    id_to_topic = {idx: topic for topic, idx in topic_to_id.items()}
    
    return topic_to_id, id_to_topic

def add_topic_embeddings_to_groups(group_df, topic, topic_emb_layer, topic_to_id, device):
    """
    Add topic embeddings to group embeddings.
    
    CRITICAL: This creates the concatenated embedding [group_emb | topic_emb] → (832,)
    
    Args:
        group_df: DataFrame with 'embedding' column (768,)
        topic: Topic name
        topic_emb_layer: nn.Embedding layer
        topic_to_id: Mapping dict
        device: torch device
    
    Returns:
        group_df: DataFrame with 'concat_embedding' column (832,)
    """
    topic_id = topic_to_id[topic]
    topic_tensor = torch.tensor([topic_id]).to(device)
    
    with torch.no_grad():
        topic_emb = topic_emb_layer(topic_tensor).squeeze(0).cpu().numpy()  # (64,)
    
    concat_embeddings = []
    
    for group_emb in group_df['embedding']:
        # Concatenate: [768] + [64] = [832]
        concat_emb = np.concatenate([group_emb, topic_emb])
        concat_embeddings.append(concat_emb)
    
    group_df['concat_embedding'] = concat_embeddings
    
    print(f"  Added topic embeddings: {group_df['concat_embedding'].iloc[0].shape}")
    
    return group_df

print("✅ Topic embedding functions defined")

## 3.3: Window Creation

In [ ]:
def create_windows(embeddings, window_size=3, stride=1, expected_dim=None):
    """
    Create sliding windows from embedding sequence.
    
    Args:
        embeddings: List or array of embeddings
        window_size: Number of timesteps per window
        stride: Step size for sliding window
        expected_dim: Expected embedding dimension (for validation)
    
    Returns:
        windows: numpy array of shape (N, window_size, dim)
        indices: Starting indices of each window
    """
    # Dimension validation
    if expected_dim is not None and len(embeddings) > 0:
        actual_dim = embeddings[0].shape[-1]
        if actual_dim != expected_dim:
            raise ValueError(f"Expected {expected_dim}D embeddings, got {actual_dim}D")
    
    if len(embeddings) < window_size:
        print(f"⚠️  Sequence too short ({len(embeddings)} < {window_size}), padding...")
        padding = [np.zeros_like(embeddings[0]) for _ in range(window_size - len(embeddings))]
        embeddings = list(embeddings) + padding
    
    windows = []
    indices = []
    
    for i in range(0, len(embeddings) - window_size + 1, stride):
        window = embeddings[i:i+window_size]
        windows.append(np.stack(window))
        indices.append(i)
    
    windows = np.stack(windows)
    
    return windows, indices

def create_windows_from_dataframe(df, embedding_col='embedding', window_size=3, stride=1, expected_dim=None):
    """
    Create windows from DataFrame with embeddings.
    
    Args:
        df: DataFrame with embedding column
        embedding_col: Name of embedding column
        window_size: Window size
        stride: Stride
        expected_dim: Expected embedding dimension
    
    Returns:
        windows: (N, window_size, dim)
        window_metadata: List of dicts with metadata
    """
    embeddings = df[embedding_col].tolist()
    windows, indices = create_windows(embeddings, window_size, stride, expected_dim)
    
    # Create metadata
    window_metadata = []
    for idx in indices:
        metadata = {
            'start_idx': idx,
            'end_idx': idx + window_size - 1,
            'dates': df['date'].iloc[idx:idx+window_size].tolist() if 'date' in df.columns else None
        }
        window_metadata.append(metadata)
    
    return windows, window_metadata

print("✅ Window creation functions defined")

## 3.4: Create Consecutive Pairs

In [ ]:
def create_consecutive_pairs(windows, topics, metadata):
    """
    Create pairs of consecutive windows for temporal contrastive learning.
    
    Args:
        windows: (N, window_size, dim)
        topics: list of topics (length N)
        metadata: list of metadata dicts
    
    Returns:
        paired_windows_current: (M, window_size, dim)
        paired_windows_next: (M, window_size, dim)
        paired_topics: list of topics
    """
    pairs_current = []
    pairs_next = []
    pairs_topics = []
    
    for i in range(len(windows) - 1):
        # Only pair windows from same topic
        if topics[i] == topics[i + 1]:
            pairs_current.append(windows[i])
            pairs_next.append(windows[i + 1])
            pairs_topics.append(topics[i])
    
    if len(pairs_current) == 0:
        print("⚠️  No valid consecutive pairs found!")
        return None, None, None
    
    paired_windows_current = np.stack(pairs_current)
    paired_windows_next = np.stack(pairs_next)
    
    print(f"✅ Created {len(pairs_current)} consecutive window pairs")
    
    return paired_windows_current, paired_windows_next, pairs_topics

print("✅ Consecutive pairs function defined")

---
# STAGE 4: Process All Topics → Unified Dataset
---

**Execute the complete preprocessing pipeline for all 5 topics.**

For each topic:
1. Load CSV
2. Extract entities
3. Compute entity embeddings
4. Entity-invariant cleaning
5. Day-level aggregation
6. Ruptures grouping
7. Group-level pooling
8. Add topic embeddings (768D → 832D)
9. Create windows

Then merge all topics into unified dataset.

In [ ]:
def process_all_topics_unified(config, sbert_model):
    """
    Process all topics and prepare for unified training.
    
    Returns:
        all_windows: Combined windows from all topics (N, window_size, 832)
        all_topics: Topic labels for each window
        all_metadata: Window metadata
        topic_emb_layer: Topic embedding layer
        topic_to_id: Topic mapping
    """
    print("="*80)
    print("UNIFIED MULTI-TOPIC PROCESSING")
    print("="*80)
    
    # Create topic embedding layer
    topic_to_id, id_to_topic = create_topic_mapping(config.TOPICS)
    num_topics = len(config.TOPICS)
    
    topic_emb_layer = nn.Embedding(num_topics, config.TOPIC_EMB_DIM)
    nn.init.xavier_uniform_(topic_emb_layer.weight)
    topic_emb_layer = topic_emb_layer.to(config.DEVICE)
    
    print(f"\n✅ Created topic embedding layer: {num_topics} topics → {config.TOPIC_EMB_DIM}D")
    print(f"   Topic mapping: {topic_to_id}")
    
    # Storage for all topics
    all_windows_list = []
    all_topics_list = []
    all_metadata_list = []
    
    # Process each topic
    for topic in config.TOPICS:
        print(f"\n{'='*80}")
        print(f"Processing: {topic}")
        print(f"{'='*80}")
        
        try:
            # 1. Load CSV
            df = load_topic_csv(topic, config.DATA_DIR)
            
            # 2. Extract entities
            df = extract_entities_batch(df, batch_size=config.NER_BATCH_SIZE)
            
            # 3. Entity embeddings
            df = compute_entity_embeddings(df, sbert_model)
            
            # 4. Entity-invariant embeddings
            method = 'projection' if config.USE_PROJECTION else 'subtraction'
            df = compute_entity_invariant_embeddings(df, method=method, lambda_=config.ENTITY_LAMBDA)
            
            # 5. Day-level aggregation
            day_df = aggregate_to_day_level(df, topic)
            
            # 6. Rupture detection (grouping)
            if config.USE_RUPTURES:
                day_df = detect_ruptures(
                    day_df,
                    model=config.RUPTURE_MODEL,
                    pen=config.RUPTURE_PEN,
                    min_size=config.MIN_GROUP_SIZE
                )
                
                # 7. Group-level aggregation
                print(f"  Aggregating to group-level...")
                group_records = []
                
                for group_id, group in day_df.groupby('group'):
                    embeddings = np.stack(group['embedding'].values)
                    group_emb = embeddings.mean(axis=0)  # Mean pooling
                    
                    # Normalize
                    norm = np.linalg.norm(group_emb)
                    if norm > 1e-6:
                        group_emb = group_emb / norm
                    
                    group_records.append({
                        'group_id': group_id,
                        'embedding': group_emb,
                        'num_days': len(group),
                        'start_date': group['date'].min(),
                        'end_date': group['date'].max(),
                        'date': group['date'].min()  # For window metadata
                    })
                
                group_df = pd.DataFrame(group_records)
                print(f"  Created {len(group_df)} groups")
            else:
                # No grouping - use days directly
                group_df = day_df.copy()
                group_df['group_id'] = range(len(group_df))
            
            # 8. *** ADD TOPIC EMBEDDINGS *** (CRITICAL)
            print(f"  Adding topic embeddings...")
            group_df = add_topic_embeddings_to_groups(
                group_df, topic, topic_emb_layer, topic_to_id, config.DEVICE
            )
            
            # 9. Create windows from concatenated embeddings
            windows, window_metadata = create_windows_from_dataframe(
                group_df,
                embedding_col='concat_embedding',  # Use 832D embeddings
                window_size=config.WINDOW_SIZE,
                stride=config.WINDOW_STRIDE,
                expected_dim=config.CONCAT_DIM  # Validate 832D
            )
            
            print(f"  Windows shape: {windows.shape}")
            
            # Store
            all_windows_list.append(windows)
            all_topics_list.extend([topic] * len(windows))
            all_metadata_list.extend(window_metadata)
            
            print(f"✅ {topic}: {len(windows)} windows")
        
        except Exception as e:
            print(f"❌ Failed to process {topic}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # Merge all topics
    print(f"\n{'='*80}")
    print("MERGING ALL TOPICS")
    print(f"{'='*80}")
    
    all_windows = np.concatenate(all_windows_list, axis=0)
    
    print(f"Total windows: {len(all_windows):,}")
    print(f"Shape: {all_windows.shape}")
    print(f"\nTopics distribution:")
    topic_counts = Counter(all_topics_list)
    for topic, count in topic_counts.items():
        print(f"  {topic}: {count:,} ({count/len(all_topics_list)*100:.1f}%)")
    
    return all_windows, all_topics_list, all_metadata_list, topic_emb_layer, topic_to_id

print("✅ Unified processing function defined")

## 4.1: Initialize SBERT and Process

In [ ]:
# Initialize SBERT model
print("\n" + "="*80)
print("INITIALIZING SBERT MODEL")
print("="*80)

sbert_model = SentenceTransformer(config.SBERT_MODEL)
sbert_model = sbert_model.to(config.DEVICE)

print(f"✅ SBERT model loaded: {config.SBERT_MODEL}")
print(f"   Embedding dimension: {config.EMBEDDING_DIM}D")
print(f"   Device: {config.DEVICE}")

In [ ]:
# EXECUTE: Process all topics
all_windows, all_topics, all_metadata, topic_emb_layer, topic_to_id = process_all_topics_unified(
    config, sbert_model
)

print("\n" + "="*80)
print("STAGE 4 COMPLETE: All topics processed")
print("="*80)
print(f"Total windows: {len(all_windows):,}")
print(f"Window shape: {all_windows.shape}")
print(f"Expected shape: (N, {config.WINDOW_SIZE}, {config.CONCAT_DIM})")

---
# STAGE 5: Dataset + DataLoader
---

Create PyTorch Dataset and DataLoader with balanced sampling.

## 5.1: Create Consecutive Pairs

In [ ]:
# Create consecutive window pairs for temporal contrastive learning
print("Creating consecutive window pairs...")

windows_current, windows_next, pair_topics = create_consecutive_pairs(
    all_windows, all_topics, all_metadata
)

if windows_current is None:
    raise ValueError("Failed to create window pairs!")

print(f"\n✅ Consecutive pairs created:")
print(f"   Current windows: {windows_current.shape}")
print(f"   Next windows: {windows_next.shape}")
print(f"   Pairs: {len(pair_topics):,}")

## 5.2: Dataset Class

In [ ]:
class PairedWindowDataset(Dataset):
    """
    Dataset for paired windows (current, next) with topic labels.
    """
    def __init__(self, windows_current, windows_next, topics, topic_to_id):
        self.windows_current = torch.tensor(windows_current, dtype=torch.float32)
        self.windows_next = torch.tensor(windows_next, dtype=torch.float32)
        self.topics = topics
        self.topic_to_id = topic_to_id
        
        # Convert topics to indices
        self.topic_indices = torch.tensor(
            [topic_to_id[t] for t in topics], 
            dtype=torch.long
        )
    
    def __len__(self):
        return len(self.windows_current)
    
    def __getitem__(self, idx):
        return {
            'window_current': self.windows_current[idx],
            'window_next': self.windows_next[idx],
            'topic_idx': self.topic_indices[idx]
        }

print("✅ Dataset class defined")

## 5.3: Balanced Topic Sampler

In [ ]:
class BalancedTopicSampler(Sampler):
    """
    Sampler that ensures balanced representation of topics.
    
    NOTE: This implementation interleaves topics for balanced batches.
    """
    def __init__(self, topics, shuffle=True):
        self.shuffle = shuffle
        
        # Group indices by topic
        self.topic_to_indices = {}
        for idx, topic in enumerate(topics):
            if topic not in self.topic_to_indices:
                self.topic_to_indices[topic] = []
            self.topic_to_indices[topic].append(idx)
        
        self.num_topics = len(self.topic_to_indices)
    
    def __iter__(self):
        # Shuffle indices within each topic
        topic_iters = {}
        for topic, indices in self.topic_to_indices.items():
            if self.shuffle:
                indices = np.random.permutation(indices).tolist()
            topic_iters[topic] = indices
        
        # Interleave samples from all topics
        all_indices = []
        max_len = max(len(indices) for indices in topic_iters.values())
        
        for i in range(max_len):
            for topic, indices in topic_iters.items():
                if i < len(indices):
                    all_indices.append(indices[i])
        
        # Shuffle the final interleaved list
        if self.shuffle:
            np.random.shuffle(all_indices)
        
        return iter(all_indices)
    
    def __len__(self):
        return sum(len(indices) for indices in self.topic_to_indices.values())

print("✅ Balanced sampler defined")

## 5.4: Create Dataset and DataLoader

In [ ]:
# Create dataset
dataset = PairedWindowDataset(
    windows_current, 
    windows_next, 
    pair_topics,
    topic_to_id
)

# Create sampler
sampler = BalancedTopicSampler(pair_topics, shuffle=True)

# Create DataLoader
train_loader = DataLoader(
    dataset,
    batch_size=config.BATCH_SIZE,
    sampler=sampler,
    num_workers=0,  # Set to 0 to avoid multiprocessing errors in Jupyter
    pin_memory=True
)

print("="*80)
print("STAGE 5 COMPLETE: Dataset and DataLoader created")
print("="*80)
print(f"Dataset size: {len(dataset):,} samples")
print(f"Batch size: {config.BATCH_SIZE}")
print(f"Num batches: {len(train_loader)}")
print(f"Sampler: BalancedTopicSampler")
print("="*80)

---
# STAGE 6: Model Architecture
---

Temporal Transformer model:
- Input: (B, 3, 832) - batches of 3-timestep windows with 832D embeddings
- Output: (B, 256) - temporal embeddings for contrastive learning

## 6.1: Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    """Add positional information to input embeddings."""
    
    def __init__(self, d_model, max_len=10):
        super().__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x: (B, T, D)
        return x + self.pe[:x.size(1), :].unsqueeze(0)

print("✅ Positional encoding defined")

## 6.2: Temporal Transformer

In [ ]:
class TemporalTransformer(nn.Module):
    """
    Transformer-based temporal encoder.
    
    Input: (B, window_size, 832)
    Output: (B, output_dim)
    """
    def __init__(self, input_dim=832, hidden_dim=512, output_dim=256,
                 num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        # Project input to hidden dimension
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(hidden_dim, max_len=10)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)        
        # Output projection
        self.output_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )
        
        # Layer norm
        self.layer_norm = nn.LayerNorm(output_dim)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: (B, T, input_dim)
            mask: optional attention mask
        
        Returns:
            output: (B, output_dim)
        """
        # Input projection
        x = self.input_proj(x)  # (B, T, hidden_dim)
        
        # Add positional encoding
        x = self.pos_encoder(x)
        
        # Transformer encoding
        x = self.transformer(x, src_key_padding_mask=mask)  # (B, T, hidden_dim)
        
        # Global pooling (mean over time)
        x = x.mean(dim=1)  # (B, hidden_dim)
        
        # Output projection
        x = self.output_proj(x)  # (B, output_dim)
        
        # Normalize
        x = self.layer_norm(x)
        x = F.normalize(x, p=2, dim=-1)  # L2 normalize for contrastive learning
        
        return x

print("✅ Temporal Transformer model defined")

## 6.3: Initialize Model

In [ ]:
# Initialize model
model = TemporalTransformer(
    input_dim=config.CONCAT_DIM,  # 832
    hidden_dim=config.HIDDEN_DIM,  # 512
    output_dim=config.OUTPUT_DIM,  # 256
    num_heads=config.NUM_HEADS,  # 8
    num_layers=config.NUM_LAYERS,  # 4
    dropout=config.DROPOUT  # 0.1
)

model = model.to(config.DEVICE)

# Model stats
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("="*80)
print("STAGE 6 COMPLETE: Model initialized")
print("="*80)
print(f"Model: TemporalTransformer")
print(f"Input dim: {config.CONCAT_DIM}D")
print(f"Hidden dim: {config.HIDDEN_DIM}D")
print(f"Output dim: {config.OUTPUT_DIM}D")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Device: {config.DEVICE}")
print("="*80)

---
# STAGE 7: Loss Functions
---

Multi-objective loss:
1. **Temporal contrastive loss** (NT-Xent): Learn temporal progression
2. **Topic separation loss**: Separate different topics in embedding space
3. **Hard negative mining**: Focus on difficult negative samples

## 7.1: NT-Xent Loss (Temporal Contrastive)

In [ ]:
class NTXentLoss(nn.Module):
    """
    Normalized Temperature-scaled Cross Entropy Loss (NT-Xent).
    Used for contrastive learning.
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, z_i, z_j):
        """
        Args:
            z_i: embeddings of anchor, shape (B, D)
            z_j: embeddings of positive, shape (B, D)
        
        Returns:
            loss: scalar
        """
        batch_size = z_i.size(0)
        
        # Concatenate
        z = torch.cat([z_i, z_j], dim=0)  # (2B, D)
        
        # Compute similarity matrix
        sim_matrix = torch.mm(z, z.t()) / self.temperature  # (2B, 2B)
        
        # Create labels: positive pairs are (i, i+B) and (i+B, i)
        # FIXED: Correct label assignment
        labels = torch.cat([
            torch.arange(batch_size) + batch_size,  # First B samples → targets at B to 2B-1
            torch.arange(batch_size)                # Last B samples → targets at 0 to B-1
        ]).to(z.device)
        
        # Mask out self-similarity
        mask = torch.eye(2 * batch_size, dtype=torch.bool).to(z.device)
        # Use -1e4 instead of -1e9 for FP16 compatibility on GPU
        sim_matrix = sim_matrix.masked_fill(mask, -1e4)
        
        # Compute loss
        loss = F.cross_entropy(sim_matrix, labels)
        
        return loss

print("✅ NT-Xent loss defined (FIXED)")

## 7.2: Topic Separation Loss (FIXED)

In [ ]:
class TopicSeparationLoss(nn.Module):
    """
    Encourage separation between different topics using L2 distance.
    
    FIXED: Uses L2 distance instead of cosine similarity.
    """
    def __init__(self, margin=0.5):
        super().__init__()
        self.margin = margin  # Minimum distance between different topics
    
    def forward(self, embeddings, topic_indices):
        """
        Args:
            embeddings: (B, D)
            topic_indices: (B,) topic indices
        
        Returns:
            loss: scalar
        """
        batch_size = embeddings.size(0)
        
        # Compute pairwise L2 distances
        dist_matrix = torch.cdist(embeddings, embeddings, p=2)
        
        # Create topic mask
        topic_mask = (topic_indices.unsqueeze(0) == topic_indices.unsqueeze(1)).float()
        diff_topic_mask = 1 - topic_mask - torch.eye(batch_size).to(embeddings.device)
        
        # Loss: max(0, margin - distance) for different topics
        # Encourages distance >= margin between different topics
        loss = F.relu(self.margin - dist_matrix) * diff_topic_mask
        
        num_pairs = diff_topic_mask.sum()
        if num_pairs > 0:
            loss = loss.sum() / num_pairs
        else:
            loss = torch.tensor(0.0).to(embeddings.device)
        
        return loss

print("✅ Topic separation loss defined (FIXED)")

## 7.3: Hard Negative Mining Loss (FIXED)

In [ ]:
class HardNegativeLoss(nn.Module):
    """
    Hard negative mining: focus on difficult negative pairs.
    
    FIXED: Uses temporal pairs (z_current, z_next) instead of self-similarity.
    """
    def __init__(self, temperature=0.07, top_k=10):
        super().__init__()
        self.temperature = temperature
        self.top_k = top_k
    
    def forward(self, z_current, z_next, topic_indices):
        """
        Args:
            z_current: (B, D) - current window embeddings
            z_next: (B, D) - next window embeddings (temporal positives)
            topic_indices: (B,)
        
        Returns:
            loss: scalar
        """
        batch_size = z_current.size(0)
        
        # Compute similarities between current and all next embeddings
        sim_matrix = torch.mm(z_current, z_next.t()) / self.temperature
        
        # Positives are on the diagonal (i-th current with i-th next)
        pos_sims = torch.diag(sim_matrix)
        
        # For each sample, find hard negatives from different topics
        losses = []
        for i in range(batch_size):
            # Hard negatives: different topic, high similarity
            diff_topic_mask = topic_indices != topic_indices[i]
            
            if diff_topic_mask.sum() > 0:
                neg_sims = sim_matrix[i][diff_topic_mask]
                top_k = min(self.top_k, len(neg_sims))
                hard_neg_sims, _ = torch.topk(neg_sims, top_k)
                
                # Contrastive loss: pos vs hard negatives
                logits = torch.cat([pos_sims[i].unsqueeze(0), hard_neg_sims])
                labels = torch.zeros(1, dtype=torch.long).to(z_current.device)
                
                loss = F.cross_entropy(logits.unsqueeze(0), labels)
                losses.append(loss)
        
        if len(losses) > 0:
            return torch.stack(losses).mean()
        else:
            return torch.tensor(0.0).to(z_current.device)

print("✅ Hard negative loss defined (FIXED)")

## 7.4: Multi-Loss (Combined)

In [ ]:
class MultiLoss(nn.Module):
    """
    Combined multi-objective loss.
    """
    def __init__(self, lambda_temporal=1.0, lambda_topic_sep=0.3, lambda_hard_neg=0.5, temperature=0.07):
        super().__init__()
        
        self.lambda_temporal = lambda_temporal
        self.lambda_topic_sep = lambda_topic_sep
        self.lambda_hard_neg = lambda_hard_neg
        
        self.ntxent_loss = NTXentLoss(temperature=temperature)
        self.topic_sep_loss = TopicSeparationLoss()
        self.hard_neg_loss = HardNegativeLoss(temperature=temperature)
    
    def forward(self, z_current, z_next, topic_indices):
        """
        Args:
            z_current: embeddings of current windows (B, D)
            z_next: embeddings of next windows (B, D)
            topic_indices: topic indices (B,)
        
        Returns:
            total_loss: scalar
            loss_dict: dictionary of individual losses
        """
        # Temporal contrastive loss
        temporal_loss = self.ntxent_loss(z_current, z_next)
        
        # Topic separation loss (on current embeddings)
        topic_loss = self.topic_sep_loss(z_current, topic_indices)
        
        # Hard negative mining (on temporal pairs)
        hard_neg_loss = self.hard_neg_loss(z_current, z_next, topic_indices)
        
        # Total loss
        total_loss = (
            self.lambda_temporal * temporal_loss +
            self.lambda_topic_sep * topic_loss +
            self.lambda_hard_neg * hard_neg_loss
        )
        
        loss_dict = {
            'total': total_loss.item(),
            'temporal': temporal_loss.item(),
            'topic_sep': topic_loss.item(),
            'hard_neg': hard_neg_loss.item()
        }
        
        return total_loss, loss_dict

print("✅ Multi-loss defined")

## 7.5: Initialize Loss Function

In [ ]:
# Initialize loss function
criterion = MultiLoss(
    lambda_temporal=config.LAMBDA_TEMPORAL,  # 1.0
    lambda_topic_sep=config.LAMBDA_TOPIC_SEP,  # 0.3
    lambda_hard_neg=config.LAMBDA_HARD_NEG,  # 0.5
    temperature=config.TEMPERATURE  # 0.07
).to(config.DEVICE)

print("="*80)
print("STAGE 7 COMPLETE: Loss functions initialized")
print("="*80)
print(f"Temporal loss weight: {config.LAMBDA_TEMPORAL}")
print(f"Topic separation weight: {config.LAMBDA_TOPIC_SEP}")
print(f"Hard negative weight: {config.LAMBDA_HARD_NEG}")
print(f"Temperature: {config.TEMPERATURE}")
print("="*80)

---
# STAGE 8: Training
---

Train the unified model on all topics.

## 8.1: Training Function

In [ ]:
def train_model(model, train_loader, num_epochs, config, criterion, topic_emb_layer=None, topic_to_id=None):
    """
    Train the temporal transformer model.
    
    Args:
        model: TemporalTransformer model
        train_loader: DataLoader
        num_epochs: number of epochs
        config: Config object
        criterion: Loss function
        topic_emb_layer: Topic embedding layer (for saving)
        topic_to_id: Topic mapping (for saving)
    
    Returns:
        model: trained model
        history: training history
    """
    device = config.DEVICE
    model = model.to(device)
    
    # Optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY
    )
    
    # Learning rate scheduler
    if config.USE_COSINE_SCHEDULE:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=config.NUM_EPOCHS // 2,
            T_mult=1,
            eta_min=1e-6
        )
    else:
        scheduler = None
    
    # Mixed precision scaler
    scaler = torch.cuda.amp.GradScaler() if config.USE_AMP else None
    
    # Training history
    history = {
        'loss': [],
        'temporal_loss': [],
        'topic_sep_loss': [],
        'hard_neg_loss': [],
        'lr': []
    }
    
    print("="*80)
    print("TRAINING START")
    print("="*80)
    
    for epoch in range(num_epochs):
        model.train()
        epoch_losses = {'total': [], 'temporal': [], 'topic_sep': [], 'hard_neg': []}
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for batch_idx, batch in enumerate(pbar):
            # Get batch data
            windows_current = batch['window_current'].to(device)
            windows_next = batch['window_next'].to(device)
            topic_indices = batch['topic_idx'].to(device)
            
            optimizer.zero_grad()
            
            # Forward pass with mixed precision
            if config.USE_AMP:
                with torch.cuda.amp.autocast():
                    z_current = model(windows_current)
                    z_next = model(windows_next)
                    loss, loss_dict = criterion(z_current, z_next, topic_indices)
                
                # Backward pass
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                z_current = model(windows_current)
                z_next = model(windows_next)
                loss, loss_dict = criterion(z_current, z_next, topic_indices)
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP)
                optimizer.step()
            
            # Track losses
            for key in loss_dict:
                epoch_losses[key].append(loss_dict[key])
            
            # Update progress bar
            pbar.set_postfix({
                'loss': f"{loss.item():.4f}",
                'temp': f"{loss_dict['temporal']:.4f}",
                'topic': f"{loss_dict['topic_sep']:.4f}",
                'hard': f"{loss_dict['hard_neg']:.4f}"
            })
        
        # Scheduler step
        if scheduler is not None:
            scheduler.step()
        
        # Epoch stats
        avg_loss = np.mean(epoch_losses['total'])
        avg_temp = np.mean(epoch_losses['temporal'])
        avg_topic = np.mean(epoch_losses['topic_sep'])
        avg_hard = np.mean(epoch_losses['hard_neg'])
        current_lr = optimizer.param_groups[0]['lr']
        
        history['loss'].append(avg_loss)
        history['temporal_loss'].append(avg_temp)
        history['topic_sep_loss'].append(avg_topic)
        history['hard_neg_loss'].append(avg_hard)
        history['lr'].append(current_lr)
        
        print(f"\nEpoch {epoch+1}/{num_epochs}:")
        print(f"  Loss: {avg_loss:.4f} | Temporal: {avg_temp:.4f} | Topic: {avg_topic:.4f} | Hard: {avg_hard:.4f}")
        print(f"  LR: {current_lr:.6f}")
        
        # Save checkpoint
        if (epoch + 1) % config.SAVE_EVERY == 0:
            checkpoint_path = config.CHECKPOINT_DIR / f"checkpoint_epoch_{epoch+1}.pt"
            save_dict = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
                'history': history,
                'config': {
                    'input_dim': config.CONCAT_DIM,
                    'hidden_dim': config.HIDDEN_DIM,
                    'output_dim': config.OUTPUT_DIM,
                    'num_heads': config.NUM_HEADS,
                    'num_layers': config.NUM_LAYERS,
                    'dropout': config.DROPOUT,
                    'topic_emb_dim': config.TOPIC_EMB_DIM
                }
            }
            
            # Add topic embeddings if provided
            if topic_emb_layer is not None:
                save_dict['topic_emb_state_dict'] = topic_emb_layer.state_dict()
            if topic_to_id is not None:
                save_dict['topic_to_id'] = topic_to_id
            
            torch.save(save_dict, checkpoint_path)
            print(f"  💾 Checkpoint saved: {checkpoint_path}")
    
    print("\n" + "="*80)
    print("TRAINING COMPLETE")
    print("="*80)
    
    return model, history

print("✅ Training function defined")

## 8.2: Run Training

In [ ]:
# EXECUTE: Train the model
print("\nStarting training...\n")

trained_model, training_history = train_model(
    model=model,
    train_loader=train_loader,
    num_epochs=config.NUM_EPOCHS,
    config=config,
    criterion=criterion,
    topic_emb_layer=topic_emb_layer,  # CRITICAL: Save topic embeddings
    topic_to_id=topic_to_id  # CRITICAL: Save topic mapping
)

print("\n" + "="*80)
print("STAGE 8 COMPLETE: Training finished")
print("="*80)
print(f"Final loss: {training_history['loss'][-1]:.4f}")
print(f"Best loss: {min(training_history['loss']):.4f}")
print("="*80)

## 8.3: Save Final Model

In [ ]:
# Save final trained model
final_model_path = config.OUTPUT_DIR / 'unified_tcl_model.pth'

torch.save({
    'model_state_dict': trained_model.state_dict(),
    'topic_emb_state_dict': topic_emb_layer.state_dict(),  # CRITICAL
    'topic_to_id': topic_to_id,  # CRITICAL
    'config': {
        'input_dim': config.CONCAT_DIM,
        'hidden_dim': config.HIDDEN_DIM,
        'output_dim': config.OUTPUT_DIM,
        'num_heads': config.NUM_HEADS,
        'num_layers': config.NUM_LAYERS,
        'dropout': config.DROPOUT,
        'topic_emb_dim': config.TOPIC_EMB_DIM
    },
    'history': training_history
}, final_model_path)

print(f"\n💾 Final model saved: {final_model_path}")
print(f"   Model includes topic embeddings and mapping")

## 8.4: Plot Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Total loss
axes[0, 0].plot(training_history['loss'])
axes[0, 0].set_title('Total Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].grid(True, alpha=0.3)

# Component losses
axes[0, 1].plot(training_history['temporal_loss'], label='Temporal')
axes[0, 1].plot(training_history['topic_sep_loss'], label='Topic Sep')
axes[0, 1].plot(training_history['hard_neg_loss'], label='Hard Neg')
axes[0, 1].set_title('Component Losses')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Learning rate
axes[1, 0].plot(training_history['lr'])
axes[1, 0].set_title('Learning Rate')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('LR')
axes[1, 0].grid(True, alpha=0.3)

# Loss comparison
axes[1, 1].bar(['Temporal', 'Topic Sep', 'Hard Neg'], [
    training_history['temporal_loss'][-1],
    training_history['topic_sep_loss'][-1],
    training_history['hard_neg_loss'][-1]
])
axes[1, 1].set_title('Final Component Losses')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Training history plotted")

---
# STAGE 9: User Inference Pipeline
---

**Complete 13-step inference pipeline for user articles.**

## Pipeline Overview:

1. Load `user_articles.csv` (date, article)
2. Split articles → sentences with windowing (2 prev + current + 2 next)
3. SBERT embeddings for windowed sentences
4. Load `topic_embedding.json` (ideal embeddings per topic)
5. Compute topic weights (cosine similarity)
6. Filter sentences (topic weight > 0.35)
7. Extract entities + entity-invariant cleaning
8. Day-level weighted pooling (per topic)
9. Add learned topic embeddings (768D → 832D)
10. Create day-level windows (NO ruptures)
11. Model prediction (832D → 256D)
12. Day-level shift detection
13. Sentence-level shift detection

**Key Differences from Training:**
- Sentence-level windowing (not article-level)
- Multi-topic weight computation
- Day-level pooling (NOT article-level)
- NO ruptures grouping
- Dual-level shift detection (day + sentence)

## 9.1: Load User Articles

In [ ]:
def load_user_articles_csv(csv_path):
    """
    Load user articles from CSV.
    
    Expected columns: date, article
    
    Returns:
        df: DataFrame with date and article columns
    """
    print("="*80)
    print("STEP 1: Loading user articles from CSV")
    print("="*80)
    
    df = pd.read_csv(csv_path)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    
    print(f"✅ Loaded {len(df)} articles")
    print(f"   Date range: {df['date'].min()} to {df['date'].max()}")
    print(f"   Columns: {list(df.columns)}")
    
    return df

print("✅ Load user articles function defined")

## 9.2: Sentence Windowing

In [ ]:
import re

def split_into_sentences(text):
    """Split text into sentences using regex."""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if len(s.strip()) > 10]

def create_sentence_windows(sentences, window_size=5):
    """
    Create windowed sentences (2 prev + current + 2 next).
    
    Args:
        sentences: List of sentence strings
        window_size: Window size (default 5)
    
    Returns:
        windows: List of dicts with center sentence and window text
    """
    pad_size = window_size // 2
    windows = []
    
    # Pad sentences
    padded = [''] * pad_size + sentences + [''] * pad_size
    
    for i in range(len(sentences)):
        window_sentences = padded[i:i + window_size]
        
        windows.append({
            'center_idx': i,
            'center_sentence': sentences[i],
            'window_sentences': window_sentences,
            'window_text': ' '.join([s for s in window_sentences if s])
        })
    
    return windows

def process_articles_to_sentences(df):
    """
    Process articles DataFrame to sentence-level with windowing.
    
    Returns:
        sentence_df: DataFrame with date, article_idx, center_sentence, window_text
    """
    print("\n" + "="*80)
    print("STEP 2: Splitting articles into sentences with windowing")
    print("="*80)
    
    records = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing articles"):
        date = row['date']
        article_text = row['article']
        
        sentences = split_into_sentences(article_text)
        windows = create_sentence_windows(sentences, window_size=5)
        
        for window in windows:
            records.append({
                'date': date,
                'article_idx': idx,
                'center_sentence': window['center_sentence'],
                'window_text': window['window_text']
            })
    
    sentence_df = pd.DataFrame(records)
    
    print(f"✅ Created {len(sentence_df)} sentence windows from {len(df)} articles")
    
    return sentence_df

print("✅ Sentence windowing functions defined")

## 9.3: Compute Sentence Embeddings

In [ ]:
def compute_sentence_embeddings_inference(sentence_df, sbert_model):
    """
    Compute SBERT embeddings for windowed sentences.
    
    Returns:
        sentence_df: DataFrame with 'embedding' column added
    """
    print("\n" + "="*80)
    print("STEP 3: Computing SBERT embeddings")
    print("="*80)
    
    texts = sentence_df['window_text'].tolist()
    
    print(f"Encoding {len(texts)} sentences...")
    embeddings = sbert_model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    
    sentence_df['embedding'] = list(embeddings)
    
    print(f"✅ Embeddings computed: {embeddings.shape}")
    
    return sentence_df

print("✅ Sentence embedding function defined")

## 9.4: Load Topic Embeddings (Ideal Articles)

In [ ]:
def load_topic_embeddings_json(json_path):
    """
    Load ideal topic embeddings from JSON.
    
    Expected format:
    {
        "Health": [0.1, 0.2, ..., 0.768],
        "War": [...],
        ...
    }
    
    Returns:
        topic_embeddings: Dict of {topic_name: np.array(768,)}
    """
    print("\n" + "="*80)
    print("STEP 4: Loading topic embeddings (ideal articles)")
    print("="*80)
    
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    topic_embeddings = {
        topic: np.array(emb) for topic, emb in data.items()
    }
    
    print(f"✅ Loaded embeddings for topics: {list(topic_embeddings.keys())}")
    for topic, emb in topic_embeddings.items():
        print(f"   {topic}: {emb.shape}")
    
    return topic_embeddings

print("✅ Topic embedding loader defined")

## 9.5: Compute Topic Weights

In [ ]:
from scipy.spatial.distance import cosine

def compute_topic_weights_inference(sentence_df, topic_embeddings):
    """
    Compute topic weights using cosine similarity.
    
    For each sentence, compute similarity to all 5 topic embeddings.
    
    Returns:
        sentence_df: DataFrame with topic weight columns added
    """
    print("\n" + "="*80)
    print("STEP 5: Computing topic weights (cosine similarity)")
    print("="*80)
    
    topics = list(topic_embeddings.keys())
    
    for topic in topics:
        topic_emb = topic_embeddings[topic]
        
        weights = []
        for sent_emb in tqdm(sentence_df['embedding'], desc=f"Computing {topic} weights"):
            # Cosine similarity = 1 - cosine distance
            sim = 1 - cosine(sent_emb, topic_emb)
            weights.append(max(0, sim))  # Clamp to [0, 1]
        
        sentence_df[f'{topic}_weight'] = weights
    
    print(f"✅ Topic weights computed for: {topics}")
    
    return sentence_df

print("✅ Topic weight computation defined")

## 9.6: Filter by Topic Weight

In [ ]:
def filter_by_topic_weights_inference(sentence_df, threshold=0.35):
    """
    Filter sentences where at least one topic weight > threshold.
    
    Returns:
        filtered_df: Filtered DataFrame
    """
    print("\n" + "="*80)
    print(f"STEP 6: Filtering sentences (topic weight > {threshold})")
    print("="*80)
    
    weight_cols = [col for col in sentence_df.columns if col.endswith('_weight')]
    
    # Keep if ANY topic weight > threshold
    mask = (sentence_df[weight_cols] > threshold).any(axis=1)
    
    filtered_df = sentence_df[mask].copy().reset_index(drop=True)
    
    print(f"✅ Filtered: {len(sentence_df)} → {len(filtered_df)} sentences")
    print(f"   Removed: {len(sentence_df) - len(filtered_df)} sentences below threshold")
    
    return filtered_df

print("✅ Filter function defined")

## 9.7: Entity-Invariant Cleaning

In [ ]:
def extract_entities_inference(sentence_df, sbert_model, nlp):
    """
    Extract entities and compute entity embeddings.
    
    Returns:
        sentence_df: DataFrame with 'entity_embedding' column
    """
    print("\n" + "="*80)
    print("STEP 7a: Extracting entities (NER)")
    print("="*80)
    
    entity_embeddings = []
    
    for sent in tqdm(sentence_df['center_sentence'], desc="Extracting entities"):
        doc = nlp(sent)
        entities = [ent.text for ent in doc.ents]
        
        if entities:
            ent_embs = sbert_model.encode(entities, convert_to_numpy=True)
            entity_emb = ent_embs.mean(axis=0)
        else:
            entity_emb = np.zeros(768)
        
        entity_embeddings.append(entity_emb)
    
    sentence_df['entity_embedding'] = entity_embeddings
    
    print(f"✅ Entity embeddings computed")
    
    return sentence_df

def compute_clean_embeddings_inference(sentence_df, lambda_=0.5):
    """
    Compute entity-invariant embeddings: E_clean = E_sem - λ * E_ent
    
    Returns:
        sentence_df: DataFrame with 'clean_embedding' column
    """
    print("\n" + "="*80)
    print(f"STEP 7b: Entity-invariant cleaning (λ={lambda_})")
    print("="*80)
    
    clean_embeddings = []
    
    for _, row in sentence_df.iterrows():
        sem_emb = row['embedding']
        ent_emb = row['entity_embedding']
        
        clean_emb = sem_emb - lambda_ * ent_emb
        
        # L2 normalize
        norm = np.linalg.norm(clean_emb)
        if norm > 1e-6:
            clean_emb = clean_emb / norm
        
        clean_embeddings.append(clean_emb)
    
    sentence_df['clean_embedding'] = clean_embeddings
    
    print(f"✅ Entity-invariant embeddings computed")
    
    return sentence_df

print("✅ Entity-invariant cleaning functions defined")

## 9.8: Day-Level Weighted Pooling

In [ ]:
def aggregate_to_day_level_inference(sentence_df):
    """
    Aggregate sentences to day-level using topic-specific weighted pooling.
    
    Creates one embedding per (date, topic) combination.
    
    Returns:
        day_df: DataFrame with date, topic, embedding columns
    """
    print("\n" + "="*80)
    print("STEP 8: Day-level weighted pooling (per topic)")
    print("="*80)
    
    topics = ['Health', 'War', 'Technology', 'Climate', 'Economics']
    
    records = []
    
    for date, group in tqdm(sentence_df.groupby('date'), desc="Day aggregation"):
        for topic in topics:
            weight_col = f'{topic}_weight'
            
            if weight_col not in group.columns:
                continue
            
            embeddings = np.stack(group['clean_embedding'].values)
            weights = group[weight_col].values
            
            # Weighted average
            weighted_sum = (embeddings.T * weights).T.sum(axis=0)
            weight_sum = weights.sum()
            
            if weight_sum > 0:
                day_embedding = weighted_sum / weight_sum
            else:
                day_embedding = embeddings.mean(axis=0)
            
            # L2 normalize
            norm = np.linalg.norm(day_embedding)
            if norm > 1e-6:
                day_embedding = day_embedding / norm
            
            records.append({
                'date': date,
                'topic': topic,
                'embedding': day_embedding,
                'num_sentences': len(group)
            })
    
    day_df = pd.DataFrame(records)
    day_df = day_df.sort_values(['topic', 'date']).reset_index(drop=True)
    
    print(f"✅ Day-level aggregation complete")
    print(f"   Total day-topic combinations: {len(day_df)}")
    print(f"   Days: {day_df['date'].nunique()}")
    print(f"   Topics: {day_df['topic'].nunique()}")
    
    return day_df

print("✅ Day-level pooling function defined")

## 9.9: Add Topic Embeddings

In [ ]:
def add_topic_embeddings_inference(day_df, topic_emb_layer, topic_to_id, device):
    """
    Add learned topic embeddings to day embeddings.
    
    768D → 832D (768D + 64D)
    
    Returns:
        day_df: DataFrame with 'embedding_with_topic' column (832D)
    """
    print("\n" + "="*80)
    print("STEP 9: Adding learned topic embeddings (768D → 832D)")
    print("="*80)
    
    embeddings_with_topic = []
    
    topic_emb_layer.eval()
    
    for _, row in day_df.iterrows():
        topic = row['topic']
        day_emb = row['embedding']
        
        topic_id = topic_to_id[topic]
        topic_emb = topic_emb_layer(torch.tensor([topic_id]).to(device)).detach().cpu().numpy()[0]
        
        combined_emb = np.concatenate([day_emb, topic_emb])
        
        embeddings_with_topic.append(combined_emb)
    
    day_df['embedding_with_topic'] = embeddings_with_topic
    
    print(f"✅ Topic embeddings added")
    print(f"   Shape: 768D + 64D = 832D")
    
    return day_df

print("✅ Topic embedding addition function defined")

## 9.10: Create Day-Level Windows

In [ ]:
def create_day_windows_inference(day_df, window_size=3):
    """
    Create windows from day-level embeddings (NO ruptures grouping).
    
    Returns:
        windows: np.array of shape (N, window_size, 832)
        window_metadata: List of dicts with date, topic info
    """
    print("\n" + "="*80)
    print(f"STEP 10: Creating day-level windows (size={window_size}, NO grouping)")
    print("="*80)
    
    topics = day_df['topic'].unique()
    
    all_windows = []
    all_metadata = []
    
    for topic in topics:
        topic_df = day_df[day_df['topic'] == topic].sort_values('date').reset_index(drop=True)
        
        embeddings = np.stack(topic_df['embedding_with_topic'].values)
        
        for i in range(len(embeddings) - window_size + 1):
            window = embeddings[i:i+window_size]
            
            all_windows.append(window)
            all_metadata.append({
                'topic': topic,
                'start_date': topic_df.iloc[i]['date'],
                'end_date': topic_df.iloc[i+window_size-1]['date'],
                'window_idx': i
            })
    
    windows = np.array(all_windows)
    
    print(f"✅ Windows created: {windows.shape}")
    print(f"   Per topic:")
    for topic in topics:
        topic_count = sum(1 for m in all_metadata if m['topic'] == topic)
        print(f"   - {topic}: {topic_count} windows")
    
    return windows, all_metadata

print("✅ Day-level windowing function defined")

## 9.11: Model Prediction

In [ ]:
def predict_with_model_inference(windows, model, device, batch_size=32):
    """
    Run model forward pass on windows.
    
    Returns:
        embeddings: np.array of shape (N, 256)
    """
    print("\n" + "="*80)
    print("STEP 11: Model prediction")
    print("="*80)
    
    model.eval()
    
    all_embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(windows), batch_size), desc="Predicting"):
            batch = windows[i:i+batch_size]
            batch_tensor = torch.tensor(batch, dtype=torch.float32).to(device)
            
            embeddings = model(batch_tensor)
            
            all_embeddings.append(embeddings.cpu().numpy())
    
    embeddings = np.concatenate(all_embeddings, axis=0)
    
    print(f"✅ Predictions complete: {embeddings.shape}")
    
    return embeddings

print("✅ Model prediction function defined")

## 9.12: Day-Level Shift Detection

In [ ]:
def detect_day_level_shifts_inference(embeddings, window_metadata, threshold_percentile=90):
    """
    Detect narrative shifts at day level using L2 distance.
    
    Returns:
        shifts: List of dicts with shift information
        all_scores: List of all shift scores
    """
    print("\n" + "="*80)
    print("STEP 12: Day-level narrative shift detection")
    print("="*80)
    
    topics = list(set(m['topic'] for m in window_metadata))
    
    all_shifts = []
    all_shift_scores = []
    
    for topic in topics:
        topic_indices = [i for i, m in enumerate(window_metadata) if m['topic'] == topic]
        topic_embeddings = embeddings[topic_indices]
        topic_metadata = [window_metadata[i] for i in topic_indices]
        
        # Compute shift scores (L2 distance between consecutive windows)
        shift_scores = []
        for i in range(len(topic_embeddings) - 1):
            dist = np.linalg.norm(topic_embeddings[i+1] - topic_embeddings[i])
            shift_scores.append(dist)
        
        all_shift_scores.extend(shift_scores)
        
        if len(shift_scores) > 0:
            threshold = np.percentile(shift_scores, threshold_percentile)
            
            for i, score in enumerate(shift_scores):
                if score > threshold:
                    all_shifts.append({
                        'topic': topic,
                        'window_idx': i,
                        'start_date': topic_metadata[i]['start_date'],
                        'end_date': topic_metadata[i]['end_date'],
                        'next_start_date': topic_metadata[i+1]['start_date'],
                        'shift_score': score,
                        'threshold': threshold,
                        'embedding_before': topic_embeddings[i],
                        'embedding_after': topic_embeddings[i+1]
                    })
    
    print(f"✅ Day-level shifts detected: {len(all_shifts)}")
    if len(all_shift_scores) > 0:
        print(f"   Threshold (P{threshold_percentile}): {np.percentile(all_shift_scores, threshold_percentile):.4f}")
    
    return all_shifts, all_shift_scores

print("✅ Day-level shift detection function defined")

## 9.13: Sentence-Level Shift Detection

In [ ]:
def detect_sentence_level_shifts_inference(day_shifts, sentence_df, model, topic_emb_layer, topic_to_id, device, top_k=5):
    """
    For high-shift days, find which sentences contributed to the shift.
    
    Returns:
        sentence_shifts: List of dicts with:
            - topic, date, sentence
            - shift_contribution
            - similarity_to_before, similarity_to_after
            - context_before, context_after (sentences)
    """
    print("\n" + "="*80)
    print("STEP 13: Sentence-level shift detection")
    print("="*80)
    
    model.eval()
    sentence_shifts = []
    
    for day_shift in tqdm(day_shifts, desc="Analyzing sentence shifts"):
        topic = day_shift['topic']
        start_date = day_shift['start_date']
        
        emb_before = day_shift['embedding_before']  # (256,) - model output
        emb_after = day_shift['embedding_after']    # (256,) - model output
        
        # Get sentences from the shift day
        day_sentences = sentence_df[
            (sentence_df['date'] == start_date) & 
            (sentence_df[f'{topic}_weight'] > 0.35)
        ].copy()
        
        if len(day_sentences) == 0:
            continue
        
        # Sort by original index to get context
        day_sentences = day_sentences.sort_index()
        
        # Get topic embedding
        topic_id = topic_to_id[topic]
        topic_emb = topic_emb_layer(torch.tensor([topic_id]).to(device)).detach().cpu().numpy()[0]  # (64,)
        
        for idx, (i, sent) in enumerate(day_sentences.iterrows()):
            # Get clean embedding (768D) and add topic embedding -> (832D)
            sent_clean_emb = sent['clean_embedding']  # (768,)
            sent_with_topic = np.concatenate([sent_clean_emb, topic_emb])  # (832,)
            
            # Pass through model to get (256D) embedding
            with torch.no_grad():
                # Create a dummy window (repeat sentence 3 times for window_size=3)
                sent_window = np.stack([sent_with_topic] * 3)  # (3, 832)
                sent_window_tensor = torch.tensor(sent_window, dtype=torch.float32).unsqueeze(0).to(device)  # (1, 3, 832)
                sent_emb_model = model(sent_window_tensor).cpu().numpy()[0]  # (256,)
            
            # Compute distances in model space (256D)
            dist_to_before = np.linalg.norm(sent_emb_model - emb_before)
            dist_to_after = np.linalg.norm(sent_emb_model - emb_after)
            
            shift_contrib = dist_to_after - dist_to_before
            
            # Get context (2 sentences before and after)
            context_before = []
            context_after = []
            
            if idx > 0:
                context_before = day_sentences.iloc[max(0, idx-2):idx]['center_sentence'].tolist()
            if idx < len(day_sentences) - 1:
                context_after = day_sentences.iloc[idx+1:min(len(day_sentences), idx+3)]['center_sentence'].tolist()
            
            sentence_shifts.append({
                'topic': topic,
                'date': start_date,
                'sentence': sent['center_sentence'],
                'shift_contribution': shift_contrib,
                'similarity_to_before': 1 / (1 + dist_to_before),
                'similarity_to_after': 1 / (1 + dist_to_after),
                'day_shift_score': day_shift['shift_score'],
                'context_before': context_before,
                'context_after': context_after
            })
    
    # Sort by absolute shift contribution
    sentence_shifts = sorted(
        sentence_shifts, 
        key=lambda x: abs(x['shift_contribution']), 
        reverse=True
    )
    
    # Keep top_k per day-topic
    final_shifts = []
    seen = {}
    
    for shift in sentence_shifts:
        key = (shift['topic'], shift['date'])
        if key not in seen:
            seen[key] = 0
        
        if seen[key] < top_k:
            final_shifts.append(shift)
            seen[key] += 1
    
    print(f"✅ Sentence-level shifts analyzed: {len(final_shifts)} top sentences")
    
    return final_shifts

print("✅ Sentence-level shift detection function defined")

## 9.14: Complete Inference Pipeline

In [ ]:
def run_complete_inference(
    user_articles_csv,
    topic_embeddings_json,
    model_path,
    output_dir,
    threshold=0.35,
    window_size=3,
    threshold_percentile=90
):
    """
    Complete user inference pipeline.
    
    Args:
        user_articles_csv: Path to user_articles.csv
        topic_embeddings_json: Path to topic_embedding.json
        model_path: Path to trained model (.pth)
        output_dir: Directory to save results
        threshold: Topic weight threshold (default 0.35)
        window_size: Day-level window size (default 3)
        threshold_percentile: Shift detection percentile (default 90)
    
    Returns:
        results: Dict with day_shifts, sentence_shifts, all_shift_scores
    """
    
    print("\n" + "="*80)
    print("COMPLETE USER INFERENCE PIPELINE")
    print("="*80)
    
    # Load model and topic embeddings
    print("\nLoading model and topic embeddings...")
    checkpoint = torch.load(model_path, map_location=config.DEVICE, weights_only=False)
    
    model = TemporalTransformer(
        input_dim=checkpoint['config']['input_dim'],
        hidden_dim=checkpoint['config']['hidden_dim'],
        output_dim=checkpoint['config']['output_dim'],
        num_heads=checkpoint['config']['num_heads'],
        num_layers=checkpoint['config']['num_layers'],
        dropout=checkpoint['config']['dropout']
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(config.DEVICE)
    
    topic_to_id = checkpoint['topic_to_id']
    num_topics = len(topic_to_id)
    topic_emb_layer = nn.Embedding(num_topics, checkpoint['config']['topic_emb_dim'])
    topic_emb_layer.load_state_dict(checkpoint['topic_emb_state_dict'])
    topic_emb_layer = topic_emb_layer.to(config.DEVICE)
    
    print("✅ Model and topic embeddings loaded")
    
    # Load SBERT and spaCy (reuse from training)
    print("\nUsing SBERT and spaCy models from training...")
    # sbert_model and nlp should already be loaded from Stage 0
    
    # Run pipeline
    df = load_user_articles_csv(user_articles_csv)
    sentence_df = process_articles_to_sentences(df)
    sentence_df = compute_sentence_embeddings_inference(sentence_df, sbert_model)
    
    topic_embeddings = load_topic_embeddings_json(topic_embeddings_json)
    sentence_df = compute_topic_weights_inference(sentence_df, topic_embeddings)
    sentence_df = filter_by_topic_weights_inference(sentence_df, threshold=threshold)
    
    sentence_df = extract_entities_inference(sentence_df, sbert_model, nlp)
    sentence_df = compute_clean_embeddings_inference(sentence_df, lambda_=0.5)
    
    day_df = aggregate_to_day_level_inference(sentence_df)
    day_df = add_topic_embeddings_inference(day_df, topic_emb_layer, topic_to_id, config.DEVICE)
    
    windows, window_metadata = create_day_windows_inference(day_df, window_size=window_size)
    
    embeddings = predict_with_model_inference(windows, model, config.DEVICE)
    day_shifts, all_scores = detect_day_level_shifts_inference(
        embeddings, window_metadata, threshold_percentile=threshold_percentile
    )
    sentence_shifts = detect_sentence_level_shifts_inference(
        day_shifts, sentence_df, model, topic_emb_layer, topic_to_id, config.DEVICE
    )
    
    # Save results
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)
    
    results = {
        'day_shifts': day_shifts,
        'sentence_shifts': sentence_shifts,
        'all_shift_scores': all_scores
    }
    
    # Save to JSON
    with open(output_dir / 'inference_results.json', 'w') as f:
        # Convert numpy arrays to lists for JSON serialization
        json_results = {
            'day_shifts': [
                {k: (v.tolist() if isinstance(v, np.ndarray) else str(v) if isinstance(v, pd.Timestamp) else v) 
                 for k, v in shift.items() if k not in ['embedding_before', 'embedding_after']}
                for shift in day_shifts
            ],
            'sentence_shifts': [
                {k: (float(v) if isinstance(v, (np.floating, np.float32, np.float64)) else 
                     str(v) if isinstance(v, pd.Timestamp) else v) 
                 for k, v in shift.items()}
                for shift in sentence_shifts
            ],
            'num_day_shifts': len(day_shifts),
            'num_sentence_shifts': len(sentence_shifts)
        }
        json.dump(json_results, f, indent=2)
    
    print("\n" + "="*80)
    print("✅ INFERENCE PIPELINE COMPLETE")
    print("="*80)
    print(f"Results saved to: {output_dir}")
    print(f"Day-level shifts: {len(day_shifts)}")
    print(f"Sentence-level shifts: {len(sentence_shifts)}")
    
    return results

print("✅ Complete inference pipeline function defined")

## 9.15: Visualization Functions

In [ ]:
def print_shift_results(results, max_display=10):
    """
    Print narrative shift results in a readable format.
    """
    day_shifts = results['day_shifts']
    sentence_shifts = results['sentence_shifts']
    
    print("\n" + "="*80)
    print("DAY-LEVEL NARRATIVE SHIFTS")
    print("="*80)
    
    for i, shift in enumerate(day_shifts[:max_display]):
        print(f"\n[{i+1}] Topic: {shift['topic']}")
        print(f"    Date: {shift['start_date']} → {shift['next_start_date']}")
        print(f"    Shift Score: {shift['shift_score']:.4f} (threshold: {shift['threshold']:.4f})")
    
    if len(day_shifts) > max_display:
        print(f"\n... and {len(day_shifts) - max_display} more")
    
    print("\n" + "="*80)
    print("SENTENCE-LEVEL NARRATIVE SHIFTS")
    print("="*80)
    
    for i, shift in enumerate(sentence_shifts[:max_display]):
        print(f"\n[{i+1}] Topic: {shift['topic']} | Date: {shift['date']}")
        print(f"    Shift Contribution: {shift['shift_contribution']:.4f}")
        print(f"    Similarity Before: {shift['similarity_to_before']:.4f}")
        print(f"    Similarity After: {shift['similarity_to_after']:.4f}")
        print(f"    Day Shift Score: {shift['day_shift_score']:.4f}")
        print(f"\n    Sentence: \"{shift['sentence']}\"")
        
        if shift['context_before']:
            print(f"\n    Context Before:")
            for j, sent in enumerate(shift['context_before']):
                print(f"      [{j-len(shift['context_before'])}] {sent}")
        
        if shift['context_after']:
            print(f"\n    Context After:")
            for j, sent in enumerate(shift['context_after']):
                print(f"      [+{j+1}] {sent}")
    
    if len(sentence_shifts) > max_display:
        print(f"\n... and {len(sentence_shifts) - max_display} more")

def plot_shift_timeline(results, figsize=(14, 6)):
    """
    Plot shift scores timeline.
    """
    all_scores = results['all_shift_scores']
    day_shifts = results['day_shifts']
    
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Timeline
    ax1 = axes[0]
    x = np.arange(len(all_scores))
    ax1.plot(x, all_scores, linewidth=1, alpha=0.7)
    
    if len(all_scores) > 0:
        threshold = np.percentile(all_scores, 90)
        ax1.axhline(threshold, color='red', linestyle='--', label=f'P90 threshold')
    
    ax1.set_xlabel('Window Index')
    ax1.set_ylabel('Shift Score (L2 Distance)')
    ax1.set_title('Narrative Shift Timeline')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Distribution
    ax2 = axes[1]
    ax2.hist(all_scores, bins=30, alpha=0.7, edgecolor='black')
    
    if len(all_scores) > 0:
        threshold = np.percentile(all_scores, 90)
        ax2.axvline(threshold, color='red', linestyle='--', label=f'P90 threshold')
    
    ax2.set_xlabel('Shift Score')
    ax2.set_ylabel('Frequency')
    ax2.set_title('Shift Score Distribution')
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Summary:")
    print(f"   Total windows: {len(all_scores)}")
    print(f"   Significant shifts: {len(day_shifts)}")
    if len(all_scores) > 0:
        print(f"   Mean shift score: {np.mean(all_scores):.4f}")
        print(f"   Max shift score: {np.max(all_scores):.4f}")

print("✅ Visualization functions defined")

## 9.16: Example Usage

In [ ]:
# Example: Run inference on user data
# Uncomment and modify paths to execute:

"""
# Run complete inference
results = run_complete_inference(
    user_articles_csv='/kaggle/input/user-data/user_articles.csv',
    topic_embeddings_json='/kaggle/input/user-data/topic_embedding.json',
    model_path=config.OUTPUT_DIR / 'unified_tcl_model.pth',
    output_dir=config.OUTPUT_DIR / 'inference_results',
    threshold=0.35,
    window_size=3,
    threshold_percentile=90
)

# Print results
print_shift_results(results, max_display=5)

# Plot timeline
plot_shift_timeline(results)
"""

print("✅ Example usage provided")
print("\n💡 To run inference:")
print("   1. Prepare user_articles.csv (columns: date, article)")
print("   2. Prepare topic_embedding.json (ideal embeddings per topic)")
print("   3. Uncomment and run the example code above")
print("   4. Results will be saved to output_dir/inference_results.json")

---
# Pipeline Complete! 🎉
---

## Summary

This notebook implements a **Temporal Contrastive Learning (TCL) pipeline** for narrative shift detection.

### Key Features:
- ✅ **Unified model** trained on 5 topics simultaneously
- ✅ **Entity-invariant embeddings** (removes entity signal bias)
- ✅ **Topic embeddings** (64D) added after grouping → 832D input
- ✅ **Multi-loss training**: Temporal + Topic separation + Hard negatives
- ✅ **Complete inference pipeline** for user articles

### Pipeline Stages:
1. **Config + Imports**: Environment setup
2. **Preprocessing**: NER, entity cleaning, day aggregation
3. **Grouping**: Ruptures detection, topic embeddings, windowing
4. **Unified Processing**: Process all 5 topics → merge
5. **Dataset**: Balanced sampling across topics
6. **Model**: Temporal Transformer (832D → 256D)
7. **Loss**: Multi-objective (temporal + topic + hard negatives)
8. **Training**: Mixed precision, gradient clipping
9. **Inference**: Article-level shift detection

### Dimension Flow:
```
Training:  SBERT (768D) → Entity cleaning (768D) → Day pooling (768D) →
           Ruptures grouping (768D) → Topic concat (832D) → Model (256D)

Inference: SBERT (768D) → Entity cleaning (768D) → Article pooling (768D) →
           Topic concat (832D) → Model (256D) → Shift scores
```

### Files Generated:
- `outputs/unified_tcl_model.pth` - Trained model + topic embeddings
- `outputs/training_history.png` - Training curves
- `checkpoints/checkpoint_epoch_*.pt` - Intermediate checkpoints

---

**Ready for Kaggle P100 GPU!** 🚀